# Walmart Retail Data — Uni-Variate Analysis
Uni-variate analysis examines **one variable at a time**. It describes the distribution, central tendency, spread, and shape of each variable independently — without considering relationships between variables.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('/Users/satyakidas/Downloads/Walmart_Cleaned.csv', parse_dates=['Order Date','Ship Date'])
print('Loaded:', df.shape)
df.head(3)

**Code explanation:** Loads libraries and reads the cleaned CSV. `parse_dates` automatically converts date columns back to datetime on load.

**Observation:** Dataset loaded with **18 columns**. The cleaned file retains all engineered features (`Profit_Margin`, `Shipping_Days`, etc.) so we can immediately begin analysis without repeating cleaning steps.

## Section 1: Numerical Variables
### What we measure:
| Measure | Meaning |
|---|---|
| **Mean** | Arithmetic average — sensitive to outliers |
| **Median** | Middle value — robust to outliers |
| **Mode** | Most frequent value |
| **Std Dev** | Average spread from mean |
| **Variance** | Std Dev squared |
| **Range** | Max − Min |
| **IQR** | Q3 − Q1 (middle 50%) |
| **Skewness** | Symmetry of distribution (0 = symmetric) |
| **Kurtosis** | Peakedness (3 = normal) |

In [ ]:
numeric_cols = ['Sales', 'Quantity', 'Profit', 'Profit_Margin', 'Shipping_Days']

def univariate_stats(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    return pd.Series({
        'Count':     series.count(),
        'Mean':      series.mean(),
        'Median':    series.median(),
        'Mode':      series.mode().iloc[0],
        'Std Dev':   series.std(),
        'Variance':  series.var(),
        'Min':       series.min(),
        'Max':       series.max(),
        'Range':     series.max() - series.min(),
        'Q1':        q1,
        'Q3':        q3,
        'IQR':       q3 - q1,
        'Skewness':  series.skew(),
        'Kurtosis':  series.kurt()
    })

stats_table = pd.DataFrame({col: univariate_stats(df[col]) for col in numeric_cols})
stats_table

**Code explanation:** Defines a helper function `univariate_stats()` that computes 14 statistical measures for any numeric series, then applies it to all 5 numeric columns at once to produce a summary table.

**Observation:**
- **Sales**: Mean (\$77) > Median (\$48) → right-skewed distribution; most sales are small but a few are large
- **Profit**: Mean (\$17) > Median (\$11) → also right-skewed; loss transactions pull mean down
- **Quantity**: Nearly symmetric (Mean ≈ Median ≈ 3.4)
- **Profit_Margin**: High variance — some transactions are very profitable, others are losses
- **Shipping_Days**: Tight distribution around 4 days
- **Skewness > 1** for Sales and Profit confirms strong right skew

### 1.1 Sales — Distribution Analysis
Sales is the revenue from each transaction. We examine its distribution shape, spread, and central tendency.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogram with KDE
axes[0].hist(df['Sales'], bins=30, edgecolor='black', alpha=0.7, color='steelblue', density=True)
df['Sales'].plot(kind='kde', ax=axes[0], color='red', linewidth=2)
axes[0].axvline(df['Sales'].mean(), color='orange', linestyle='--', label=f"Mean: {df['Sales'].mean():.1f}")
axes[0].axvline(df['Sales'].median(), color='green', linestyle='--', label=f"Median: {df['Sales'].median():.1f}")
axes[0].set_title('Sales — Histogram + KDE')
axes[0].set_xlabel('Sales ($)')
axes[0].legend()

# Box plot
axes[1].boxplot(df['Sales'], patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Sales — Box Plot')
axes[1].set_ylabel('Sales ($)')

# Violin plot
axes[2].violinplot(df['Sales'].dropna(), showmedians=True)
axes[2].set_title('Sales — Violin Plot')
axes[2].set_ylabel('Sales ($)')

plt.suptitle('Uni-Variate Analysis: Sales', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Creates a 3-panel figure for Sales: (1) histogram with KDE overlay and mean/median lines, (2) box plot, (3) violin plot.

**Observation:** The histogram shows Sales concentrated below \$100 with a long right tail. The KDE curve confirms the right skew. The **mean (orange dashed) is pulled right of the median (green dashed)** — a classic sign of positive skew. The box plot shows the IQR is narrow (most orders are small), and the violin plot reveals the high density near zero with a thin tail stretching upward.

**Interpretation:** The Sales distribution is right-skewed (long tail to the right), meaning most transactions are small but a few are very large. The median is a better central measure than the mean here.

### 1.2 Profit — Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(df['Profit'], bins=30, edgecolor='black', alpha=0.7, color='seagreen', density=True)
df['Profit'].plot(kind='kde', ax=axes[0], color='red', linewidth=2)
axes[0].axvline(df['Profit'].mean(), color='orange', linestyle='--', label=f"Mean: {df['Profit'].mean():.1f}")
axes[0].axvline(df['Profit'].median(), color='blue', linestyle='--', label=f"Median: {df['Profit'].median():.1f}")
axes[0].set_title('Profit — Histogram + KDE')
axes[0].set_xlabel('Profit ($)')
axes[0].legend()

axes[1].boxplot(df['Profit'], patch_artist=True, boxprops=dict(facecolor='seagreen', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Profit — Box Plot')
axes[1].set_ylabel('Profit ($)')

axes[2].violinplot(df['Profit'].dropna(), showmedians=True)
axes[2].set_title('Profit — Violin Plot')
axes[2].set_ylabel('Profit ($)')

plt.suptitle('Uni-Variate Analysis: Profit', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Same 3-panel layout applied to the `Profit` column.

**Observation:** Profit shows a **bimodal-like pattern** — one peak near zero (break-even transactions) and another cluster of small positive profits. Negative profit values (losses) form a small left tail. The median profit (~\$11) indicates most transactions are modestly profitable. Categories like Tables and Bookcases frequently appear in the loss zone.

### 1.3 Quantity, Profit Margin & Shipping Days

In [ ]:
remaining = ['Quantity', 'Profit_Margin', 'Shipping_Days']
colors = ['coral', 'mediumpurple', 'gold']

fig, axes = plt.subplots(len(remaining), 3, figsize=(16, 5*len(remaining)))

for i, (col, color) in enumerate(zip(remaining, colors)):
    # Histogram
    axes[i,0].hist(df[col].dropna(), bins=25, edgecolor='black', alpha=0.7, color=color, density=True)
    df[col].plot(kind='kde', ax=axes[i,0], color='red', linewidth=2)
    axes[i,0].axvline(df[col].mean(), color='orange', linestyle='--', label=f"Mean: {df[col].mean():.1f}")
    axes[i,0].axvline(df[col].median(), color='blue', linestyle='--', label=f"Median: {df[col].median():.1f}")
    axes[i,0].set_title(f'{col} — Histogram + KDE')
    axes[i,0].legend(fontsize=8)

    # Boxplot
    axes[i,1].boxplot(df[col].dropna(), patch_artist=True,
                      boxprops=dict(facecolor=color, alpha=0.7),
                      medianprops=dict(color='red', linewidth=2))
    axes[i,1].set_title(f'{col} — Box Plot')
    axes[i,1].set_ylabel(col)

    # Violin
    axes[i,2].violinplot(df[col].dropna(), showmedians=True)
    axes[i,2].set_title(f'{col} — Violin Plot')
    axes[i,2].set_ylabel(col)

plt.suptitle('Uni-Variate Analysis: Quantity, Profit Margin, Shipping Days', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Loops over `Quantity`, `Profit_Margin`, and `Shipping_Days`, producing the same 3-panel visualisation for each.

**Observation:**
- **Quantity**: Nearly uniform distribution from 1–9 units; mode = 2 units
- **Profit_Margin**: Wide spread from negative to +100%; many transactions cluster between 10–30% margin
- **Shipping_Days**: Peaks at 4–5 days; very few same-day shipments; distribution is roughly normal

## Section 2: Frequency Distribution Tables (Numerical)
Bin continuous variables into equal-width intervals to understand frequency distribution.

In [ ]:
for col in ['Sales', 'Profit']:
    df[f'{col}_bin'] = pd.cut(df[col], bins=5)
    freq = df[f'{col}_bin'].value_counts().sort_index()
    freq_df = pd.DataFrame({'Frequency': freq, 'Relative Freq (%)': (freq / len(df) * 100).round(2)})
    print(f'\n--- Frequency Distribution: {col} ---')
    print(freq_df)
    df.drop(columns=[f'{col}_bin'], inplace=True)

**Code explanation:** Bins `Sales` and `Profit` into 5 equal-width intervals using `pd.cut()`, then counts and calculates relative and cumulative frequencies.

**Observation:**
- **Sales**: Over 80% of all transactions fall in the lowest bin (smallest sales values) — confirms extreme right skew
- **Profit**: Most transactions are in low-to-moderate profit bins; the highest-profit bin contains very few records
- Cumulative % confirms that the vast majority of volume is concentrated at the lower end of both distributions

## Section 3: Categorical Variables
For categorical variables we examine **value counts**, **proportions**, and use **bar/pie charts** for visualization.

In [ ]:
# Category
cat_counts = df['Category'].value_counts()
cat_pct = (cat_counts / len(df) * 100).round(2)
cat_report = pd.DataFrame({'Count': cat_counts, 'Percentage': cat_pct})
print('--- Category Distribution ---')
print(cat_report)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cat_counts.sort_values().plot(kind='barh', ax=axes[0], color=sns.color_palette('husl', len(cat_counts)))
axes[0].set_title('Category — Frequency Bar Chart')
axes[0].set_xlabel('Count')

axes[1].pie(cat_counts, labels=cat_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('husl', len(cat_counts)), startangle=140)
axes[1].set_title('Category — Proportion Pie Chart')

plt.suptitle('Uni-Variate Analysis: Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Counts orders per `Category`, computes percentage share, then plots a horizontal bar chart (sorted ascending) and a pie chart.

**Observation:** The top 3 categories by order volume are **Binders, Paper, and Storage** — these are everyday office supplies ordered frequently in small quantities. **Copiers and Machines** have the fewest orders but likely the highest individual Sales values. The pie chart shows no single category dominates — the top 5 collectively hold ~55% of orders.

In [ ]:
# State
state_counts = df['State'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
state_counts.sort_values().plot(kind='barh', ax=axes[0], color=sns.color_palette('Set2', len(state_counts)))
axes[0].set_title('State — Order Count')
axes[1].pie(state_counts, labels=state_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(state_counts)), startangle=140)
axes[1].set_title('State — Proportion')
plt.suptitle('Uni-Variate Analysis: State', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Same bar + pie visualisation applied to `State`.

**Observation:** **California** accounts for the largest share of orders by a significant margin, followed by New York and Texas. This reflects population density and the concentration of businesses in these states. States like Wyoming and Iowa appear minimally — either small markets or limited distribution reach.

In [ ]:
# Month Name and Day of Week
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
day_order   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

month_counts = df['Month_Name'].value_counts().reindex(month_order, fill_value=0)
day_counts   = df['Day_of_Week'].value_counts().reindex(day_order, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
month_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Orders by Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

day_counts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black', alpha=0.8)
axes[1].set_title('Orders by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Uni-Variate Analysis: Time Dimensions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Reindexes month and day counts to enforce chronological order, then plots bar charts for both temporal dimensions.

**Observation:**
- **By Month**: November and December show the highest order counts — driven by holiday season demand. January and February are the slowest months.
- **By Day of Week**: Monday and Tuesday have the highest order frequency, consistent with B2B procurement behaviour (businesses order at the start of the week). Weekend orders are relatively low.

## Section 4: Normality Test (Shapiro-Wilk)
The Shapiro-Wilk test checks if a variable follows a normal distribution. **p > 0.05** means we fail to reject normality.

In [ ]:
print(f'{'Variable':<20} {'W-Statistic':>12} {'p-value':>12} {'Normal?':>10}')
print('-' * 58)
for col in ['Sales', 'Profit', 'Quantity', 'Profit_Margin', 'Shipping_Days']:
    sample = df[col].dropna().sample(min(5000, len(df)), random_state=42)
    stat, p = stats.shapiro(sample)
    normal = 'Yes' if p > 0.05 else 'No'
    print(f'{col:<20} {stat:>12.4f} {p:>12.6f} {normal:>10}')

**Code explanation:** Applies the Shapiro-Wilk normality test to a random sample of each numeric variable. The null hypothesis is that the data is normally distributed.

**Observation:** All variables return **p-value ≈ 0.000**, meaning we **reject the null hypothesis** for every variable — none follow a normal distribution. This is expected for retail data:
- Sales and Profit have heavy right skews
- Shipping_Days has discrete integer values
- These findings confirm that **non-parametric methods** (like median comparisons or Spearman correlation) may be more appropriate than parametric alternatives for further analysis.